<a href="https://colab.research.google.com/github/pksX01/Python-Tutorials/blob/master/Udacity_Spark_Course_L5_dataframe_quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Wrangling with DataFrames Coding Quiz

Use this Jupyter notebook to find the answers to the quiz in the previous section. There is an answer key in the next part of the lesson.

In [ ]:
from pyspark.sql import SparkSession

# TODOS:
# 1) import any other libraries you might need
# 2) instantiate a Spark session
# 3) read in the data set located at the path "data/sparkify_log_small.json"
# 4) write code to answer the quiz questions

spark = SparkSession.builder.appName("Quiz").master("local").getOrCreate()
sparkify_log = spark.read.json("data/sparkify_log_small.json")

# Question 1

Which page did user id "" (empty string) NOT visit?

In [ ]:
sparkify_log.show(2)

+-------------+---------+---------+------+-------------+--------+---------+-----+--------------------+------+--------+-------------+---------+--------------------+------+-------------+--------------------+------+
|       artist|     auth|firstName|gender|itemInSession|lastName|   length|level|            location|method|    page| registration|sessionId|                song|status|           ts|           userAgent|userId|
+-------------+---------+---------+------+-------------+--------+---------+-----+--------------------+------+--------+-------------+---------+--------------------+------+-------------+--------------------+------+
|Showaddywaddy|Logged In|  Kenneth|     M|          112|Matthews|232.93342| paid|Charlotte-Concord...|   PUT|NextSong|1509380319284|     5132|Christmas Tears W...|   200|1513720872284|"Mozilla/5.0 (Win...|  1046|
|   Lily Allen|Logged In|Elizabeth|     F|            7|   Chase|195.23873| free|Shreveport-Bossie...|   PUT|NextSong|1512718541284|     5027|      

In [ ]:
# TODO: write your code to answer question 1
pages_emptyString_userID_visit = sparkify_log.filter(sparkify_log.userId == '').select("page").dropDuplicates()
all_pages = sparkify_log.select('page').dropDuplicates()

for row in set(all_pages.collect()) - set(pages_emptyString_userID_visit.collect()):
    print(row['page'])

NextSong
Submit Upgrade
Logout
Save Settings
Downgrade
Error
Submit Downgrade
Settings
Upgrade


# Question 2 - Reflect

What type of user does the empty string user id most likely refer to?


In [ ]:
# TODO: use this space to explore the behavior of the user with an empty string
sparkify_log.where(sparkify_log["userId"] == "").show()

+------+----------+---------+------+-------------+--------+------+-----+--------+------+-----+------------+---------+----+------+-------------+---------+------+
|artist|      auth|firstName|gender|itemInSession|lastName|length|level|location|method| page|registration|sessionId|song|status|           ts|userAgent|userId|
+------+----------+---------+------+-------------+--------+------+-----+--------+------+-----+------------+---------+----+------+-------------+---------+------+
|  null|Logged Out|     null|  null|            0|    null|  null| free|    null|   PUT|Login|        null|     5598|null|   307|1513721196284|     null|      |
|  null|Logged Out|     null|  null|           26|    null|  null| paid|    null|   GET| Home|        null|      428|null|   200|1513721274284|     null|      |
|  null|Logged Out|     null|  null|            5|    null|  null| free|    null|   GET| Home|        null|     2941|null|   200|1513722009284|     null|      |
|  null|Logged Out|     null|  nul

# Question 3

How many female users do we have in the data set?

In [ ]:
# TODO: write your code to answer question 3
sparkify_log.select(["userId", "gender"]).where(sparkify_log["gender"] == "F").dropDuplicates().count()

462

# Question 4

How many songs were played from the most played artist?

In [ ]:
# TODO: write your code to answer question 4
from pyspark.sql.functions import count
sparkify_log.groupBy("artist").agg(count("song").alias("cnt")).sort("cnt", ascending=False).show(1)

+--------+---+
|  artist|cnt|
+--------+---+
|Coldplay| 83|
+--------+---+
only showing top 1 row



# Question 5 (challenge)

How many songs do users listen to on average between visiting our home page? Please round your answer to the closest integer.



In [ ]:
# TODO: write your code to answer question 5
from pyspark.sql.functions import avg, sum, udf, col
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

user_window = Window.partitionBy("userId").orderBy("ts")\
                .rangeBetween(Window.unboundedPreceding, 0)

func = udf(lambda isHome: int(isHome == 'Home'), IntegerType())

sparkify_log_with_home_page_visit_stats = sparkify_log\
            .filter((sparkify_log.page == 'NextSong') | (sparkify_log.page == 'Home'))\
            .select('userId', 'page', 'ts')\
            .withColumn('homeVisit', func(col('page')))\
            .withColumn('frequency', sum('homeVisit').over(user_window))

sparkify_log_with_home_page_visit_stats.filter(sparkify_log_with_home_page_visit_stats.page == 'NextSong')\
                    .groupBy('userId', 'frequency')\
                    .agg({'frequency' : 'count'})\
                    .agg({'count(frequency)' : 'avg'}).show()

+---------------------+
|avg(count(frequency))|
+---------------------+
|   6.9558333333333335|
+---------------------+

